# Verifiers: the one pluggable rule

Algorithm 3 leaves exactly one component abstract — `Verify`. It is where RMC and D-GRS differ, where your own coupling implementztion goes.

```python
class MyRule(Verifier):
    name = "my-rule"
    max_children = None            # or 1 for a single-proposal coupling

    def verify(self, request: VerifyRequest) -> VerifyResult:
        ...
```

That is the whole interface. A rule written this way also runs under the batched sampler
unchanged.

### The three obligations

1. `result.state` is an **exact** sample from `N(target_mean, sigma^2 I)`.
2. If `result.accepted`, then `result.state` **is** `children[child_index]` — not a copy, not a
   corrected version. The sampler descends into that child's subtree.
3. Children are examined **in the order given**, if your rule is a sequence coupling.

Obligation 2 is machine-checkable and `CheckedVerifier` checks it. Obligation 1 cannot be
checked per call — it gets a statistical test, `check_exactness`. This notebook covers both,
plus the rank-1 coordinates every rule works in.

In [1]:
import numpy as np

from specdiff import (
    CheckedVerifier,
    DraftTree,
    ResampleVerifier,
    Verifier,
    VerifyRequest,
    VerifyResult,
    available_verifiers,
    check_exactness,
    create_verifier,
    register_verifier,
)
from specdiff.ops import standard_normal_sf
from specdiff.verifiers.rank1 import Rank1Frame

rng = np.random.default_rng(0)

## 1. The request and the result

`VerifyRequest` is everything a rule is allowed to see at one node; `VerifyResult` is
everything it may say back. Normally the sampler builds the request, but building one by hand
is the fastest way to develop a rule.

In [2]:
dim, sigma, K = 6, 0.5, 3

mu_p = np.zeros(dim)                                  # m^p(Y_u): where the children were drawn
mu_q = mu_p + 0.4 * np.array([1.0, 0, 0, 0, 0, 0])    # m^q(Y_u): where they should have been
children = mu_p + sigma * rng.standard_normal((K, dim))   # the K drafted children, in order

request = VerifyRequest(
    step=7,
    proposal_mean=mu_p,
    target_mean=mu_q,
    sigma=sigma,
    children=children,
    parent_state=np.zeros(dim),
    rng=rng,
    info={"level": 1, "node": 0},
)

print("K            ", request.num_children)
print("state_shape  ", request.state_shape)
print("child(1)     ", request.child(1).round(3))
print("info         ", dict(request.info))

K             3
state_shape   (6,)
child(1)      [ 0.652  0.474 -0.352 -0.633 -0.312  0.021]
info          {'level': 1, 'node': 0}


In [3]:
request

VerifyRequest(step=7, proposal_mean=array([0., 0., 0., 0., 0., 0.]), target_mean=array([0.4, 0. , 0. , 0. , 0. , 0. ]), sigma=0.5, children=array([[ 0.06286511, -0.06605243,  0.32021133,  0.05245006, -0.26783469,
         0.18079753],
       [ 0.65200002,  0.47354048, -0.35186762, -0.63271074, -0.31163723,
         0.02066299],
       [-1.16251539, -0.10939583, -0.62295547, -0.36613368, -0.27212949,
        -0.15815008]]), parent_state=array([0., 0., 0., 0., 0., 0.]), index_in_batch=0, rng=Generator(PCG64) at 0x118909700, info={'level': 1, 'node': 0})

In [4]:
# VerifyResult refuses the inconsistent combinations at construction.
print(VerifyResult(children[0], accepted=True, child_index=0, proposals_examined=1))

for bad in (
    lambda: VerifyResult(children[0], accepted=True),                    # accepted, no index
    lambda: VerifyResult(children[0], accepted=False, child_index=0),    # rejected, with index
):
    try:
        bad()
    except ValueError as exc:
        print("ValueError ->", exc)

VerifyResult(state=array([ 0.06286511, -0.06605243,  0.32021133,  0.05245006, -0.26783469,
        0.18079753]), accepted=True, child_index=0, proposals_examined=1)
ValueError -> accepted=True requires child_index (the accepted v*)
ValueError -> child_index must be None when accepted=False


In [5]:
request.child(0)

array([ 0.06286511, -0.06605243,  0.32021133,  0.05245006, -0.26783469,
        0.18079753])

In [6]:
request.children[0]

array([ 0.06286511, -0.06605243,  0.32021133,  0.05245006, -0.26783469,
        0.18079753])

## 2. The reference rule: always resample

`ResampleVerifier` ignores the drafts and draws a fresh state from the target kernel. It is
trivially exact and trivially useless — one committed state per target call, i.e. `1.00x`. That
makes it the sampler's ground truth: if Algorithm 3 with this rule does not match a plain
Euler-Maruyama loop in distribution, the bug is in the sampler, not in the coupling.

Note the three things worth copying from it: draw from `request.rng` (never `np.random`
directly, or the run stops being reproducible from the caller's seed), return `accepted=False`
with **no** `child_index`, and get the backend from `self.backend_for(request)` so the rule
never imports NumPy or torch.

In [7]:
result = ResampleVerifier()(request)
print("accepted      ", result.accepted)
print("child_index   ", result.child_index)
print("state         ", result.state.round(3))
print("distance to mu_q (~ sigma * sqrt(d) =", round(sigma * np.sqrt(dim), 3), "):",
      round(float(np.linalg.norm(result.state - mu_q)), 3))

accepted       False
child_index    None
state          [ 0.606  0.521 -0.064  0.683 -0.333  0.176]
distance to mu_q (~ sigma * sqrt(d) = 1.225 ): 0.963


## 3. A useful rule that accepts nothing: the delta probe

Before committing to a coupling, measure the headroom. This rule records the normalised mean
mismatch `delta` at every node and then resamples, so it is exact by construction and safe to
run against a production model — and `delta` determines every acceptance probability in the
paper.

In [8]:
class DeltaProbe(Verifier):
    """Records the mean mismatch at every node, then resamples exactly."""

    name = "delta-probe"

    def __init__(self):
        self.deltas = []

    def reset(self):                       # called at the start of each sample()
        self.deltas.clear()

    def verify(self, request: VerifyRequest) -> VerifyResult:
        frame = Rank1Frame.from_request(request)
        self.deltas.append(frame.delta)

        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
        return VerifyResult(request.target_mean + request.sigma * noise, accepted=False)


probe = DeltaProbe()
probe(request)
print("delta at this node:", round(probe.deltas[0], 4),
      "  (= ||mu_q - mu_p|| / sigma =", round(float(np.linalg.norm(mu_q - mu_p)) / sigma, 4), ")")

delta at this node: 0.8   (= ||mu_q - mu_p|| / sigma = 0.8 )


## 4. Rank-1 coordinates

`P` and `Q` are isotropic Gaussians differing only in mean, so everything orthogonal to
`mu_q - mu_p` has the *same law under both*, and the `d`-dimensional coupling collapses to a
scalar one:

```
S ~ N(0, 1)        under P
S ~ N(delta, 1)    under Q          delta = ||mu_q - mu_p|| / sigma
```

A rule only has to move the scalar `S`; the orthogonal residual of whichever proposal it keeps
rides along untouched. `Rank1Frame` is that change of coordinates (eqs. 8–11).

In [9]:
frame = Rank1Frame.from_request(request)

print("delta        ", round(frame.delta, 4))
print("direction    ", frame.direction.round(3), " (unit:", round(float(np.linalg.norm(frame.direction)), 12), ")")

s, z_perp = frame.project(children[0])         # eq. (10):  Y -> (S, Z_perp)
back = frame.reconstruct(s, z_perp)            # eq. (11):  (S, Z_perp) -> Y
print("s            ", round(s, 4))
print("z_perp . e   ", round(float(z_perp @ frame.direction), 12), " (orthogonal by construction)")
print("round trip   ", np.allclose(back, children[0]))

delta         0.8
direction     [1. 0. 0. 0. 0. 0.]  (unit: 1.0 )
s             0.1257
z_perp . e    0.0  (orthogonal by construction)
round trip    True


In [10]:
# The claim the reduction rests on, checked empirically: the projection is
# N(0, 1) under the proposal kernel and N(delta, 1) under the target kernel.
n = 200_000
under_p = mu_p + sigma * rng.standard_normal((n, dim))
under_q = mu_q + sigma * rng.standard_normal((n, dim))
proj = lambda ys: ((ys - mu_p) / sigma) @ frame.direction

print(f"under P:  mean {proj(under_p).mean():+.4f}  std {proj(under_p).std():.4f}   (expect 0, 1)")
print(f"under Q:  mean {proj(under_q).mean():+.4f}  std {proj(under_q).std():.4f}   "
      f"(expect {frame.delta:.4f}, 1)")

under P:  mean +0.0029  std 1.0005   (expect 0, 1)
under Q:  mean +0.8037  std 0.9993   (expect 0.8000, 1)


### Degeneracy — check it before dividing by `delta`

`frame.degenerate` is `delta <= tol` with `tol = 1e-10` — a constant, **not** derived from the
state dtype — and **not** `delta == 0`. The reason is worth internalising: the
regime that breaks a rule is small-and-nonzero `delta`, which is exactly what a *good* proposal
produces. At `delta = 1e-16`, an `== 0` test says "not degenerate" while `tau = ln(lambda)/delta`
is ~1e15, `Phi_bar` saturates to exactly 0, and a residual mass becomes 0 — a division by zero
one step later. Below `tol` the two kernels are indistinguishable at the state's own precision,
so accepting unconditionally is the correct limit, not an approximation (Remark 2).

In [11]:
def frame_at(delta, dtype=np.float64):
    d = np.zeros(dim, dtype=dtype)
    d[0] = delta * sigma
    return Rank1Frame.from_request(
        VerifyRequest(step=0, proposal_mean=np.zeros(dim, dtype), target_mean=d,
                      sigma=sigma, children=np.zeros((1, dim), dtype))
    )


for delta in (0.0, 1e-16, 1e-6, 0.3):
    f = frame_at(delta)
    print(f"delta={delta:<8g} tol={f.tol:.2e}  degenerate={f.degenerate}")

print("float32 tol:", frame_at(0.3, np.float32).tol)

# tau raises on a degenerate frame instead of handing back an infinity.
try:
    frame_at(0.0).tau(0.5)
except ZeroDivisionError as exc:
    print("\nZeroDivisionError ->", str(exc).splitlines()[0])

print("tau(0.5) at delta=0.3:", round(frame_at(0.3).tau(0.5), 4))

delta=0        tol=1.00e-10  degenerate=True
delta=1e-16    tol=1.00e-10  degenerate=True
delta=1e-06    tol=1.00e-10  degenerate=False
delta=0.3      tol=1.00e-10  degenerate=False
float32 tol: 1e-10

ZeroDivisionError -> tau is undefined on a degenerate frame (delta=0.000e+00 <= tol=1.000e-10). Check `frame.degenerate` first and accept the first proposal: Remark 2 gives acceptance probability 1.
tau(0.5) at delta=0.3: -2.3105


## 5. Testing obligation 1: `check_exactness`

Exactness is the property the whole method is sold on, and the sampler cannot check it at
runtime: a rule that returns a plausible-looking Gaussian from the *wrong* distribution
produces a run that finishes, reports a speedup, and is silently wrong.

`check_exactness` runs the rule many times on a synthetic node and KS-tests the projection of
its output onto the displacement direction, where exactness implies `N(delta, 1)` **whatever
the rule did internally**. No SciPy needed.

In [12]:
report = check_exactness(ResampleVerifier(), delta=1.5, num_children=4, seed=0)
print(report)
print()
print("ks_statistic  ", round(report.ks_statistic, 5))
print("critical      ", round(report.critical_value, 5))
print("passed        ", report.passed)
print("acceptance    ", report.acceptance_rate, " (a resampling rule never accepts)")
print("mean_examined ", report.mean_examined)

[PASS] delta=1.500 K=4 n=4000 KS=0.0119 (crit 0.0257) accept=0.000

ks_statistic   0.01188
critical       0.02573
passed         True
acceptance     0.0  (a resampling rule never accepts)
mean_examined  0.0


Sweep the regimes that actually differ, and **include `delta = 0`**: it is the case a good
proposal approaches, the one Remark 2 forces every rule to special-case, and therefore the one
most likely to be wrong. Note the tighter `alpha` — a sweep runs many tests, so at the default
`alpha=0.01` the chance that *some* cell trips on a correct rule grows with the number of cells.

**Keep the seed.** `seed` controls every draw — the direction, the children, and whatever the
rule consumes via `request.rng` — so a report is reproducible. This is a hypothesis test at
level `alpha`, so a correct rule fails about `alpha` of the time, and an irreproducible failure
is indistinguishable from a real coupling bug.

In [13]:
def sweep(rule_factory, deltas=(0.0, 0.1, 1.0, 3.0), ks=(1, 2, 8), alpha=0.001):
    print(f"{'delta':>7} " + "".join(f"{'K=' + str(k):>22}" for k in ks))
    for delta in deltas:
        row = f"{delta:>7.1f} "
        for k in ks:
            r = check_exactness(rule_factory(), delta=delta, num_children=k, seed=0, alpha=alpha)
            row += f"{('PASS' if r.passed else 'FAIL') + f' KS={r.ks_statistic:.4f}':>22}"
        print(row)


sweep(ResampleVerifier)

  delta                    K=1                   K=2                   K=8
    0.0         PASS KS=0.0192        PASS KS=0.0122        PASS KS=0.0172


    0.1         PASS KS=0.0192        PASS KS=0.0122        PASS KS=0.0172


    1.0         PASS KS=0.0192        PASS KS=0.0122        PASS KS=0.0172
    3.0         PASS KS=0.0192        PASS KS=0.0122        PASS KS=0.0172


The KS values repeat down each column, which is not a bug in the sweep: this rule's output does
not depend on `delta` except through `mu_q`, so at a fixed seed the projected samples shift by
exactly `delta` and the statistic is unchanged. A rule that *uses* the children — any real
coupling — will not behave like that.

### What a wrong rule looks like

Two plausible-looking rules that are *not* exact. The first accepts the child nearest the target
mean — an eminently sensible-sounding heuristic, and completely wrong: it never rejects, so its
output is a nearest-order-statistic, not a Gaussian. The second is subtler and the kind of bug
you actually write: it resamples correctly but centres on the *proposal* mean, off by `delta`.

In [14]:
class NearestChild(Verifier):
    """Plausible, and wrong: an order statistic is not a Gaussian."""

    name = "nearest-child"

    def verify(self, request):
        d = ((request.children - request.target_mean) ** 2).sum(axis=1)
        k = int(np.argmin(d))
        return VerifyResult(request.child(k), accepted=True, child_index=k, proposals_examined=request.num_children)


class WrongCentre(Verifier):
    """Off by delta: resamples around mu_p instead of mu_q."""

    name = "wrong-centre"

    def verify(self, request):
        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.proposal_mean, request.rng)[0]
        return VerifyResult(request.proposal_mean + request.sigma * noise, accepted=False)


for rule in (NearestChild, WrongCentre):
    r = check_exactness(rule(), delta=1.0, num_children=4, seed=0)
    print(f"{rule.name:>14}: {r}")

 nearest-child: [FAIL] delta=1.000 K=4 n=4000 KS=0.2792 (crit 0.0257) accept=1.000


  wrong-centre: [FAIL] delta=1.000 K=4 n=4000 KS=0.3779 (crit 0.0257) accept=0.000


Both fail loudly. Note that `NearestChild` *passes* at `delta = 0` for `K = 1` — with one child
and no mismatch, "accept the only child" is exactly right. A sweep is what exposes it:

In [15]:
sweep(NearestChild)

  delta                    K=1                   K=2                   K=8


    0.0         PASS KS=0.0122        FAIL KS=0.0470        FAIL KS=0.0993


    0.1         FAIL KS=0.0520        FAIL KS=0.0634        FAIL KS=0.1129


    1.0         FAIL KS=0.3903        FAIL KS=0.3263        FAIL KS=0.2579


    3.0         FAIL KS=0.8697        FAIL KS=0.8250        FAIL KS=0.7366


## 6. Testing obligation 2: `check_contract=True`

`CheckedVerifier` costs a couple of comparisons per node and catches the checkable half:
wrong result type, wrong state shape, non-finite values, an out-of-range `child_index`, and the
important one — `accepted=True` with a state that is not the drafted child. Pass
`check_contract=True` to either sampler and it wraps your rule for you.

It is **not** an exactness check: `NearestChild` above sails through it.

In [16]:
class ContractBreaker(Verifier):
    """Accepts, but returns a *corrected* state rather than the child itself."""

    name = "contract-breaker"

    def verify(self, request):
        nudged = request.child(0) + 1e-3 * (request.target_mean - request.proposal_mean)
        return VerifyResult(nudged, accepted=True, child_index=0)


checked = CheckedVerifier(ContractBreaker())
try:
    checked(request)
except ValueError as exc:
    print("ValueError ->", str(exc))

print()
print("NearestChild under the contract checker:",
      CheckedVerifier(NearestChild())(request).accepted, "<- exactness is not what this checks")

ValueError -> contract-breaker reported accepted=True but the returned state is not child 0. An accepted state must be the drafted state itself, otherwise the sampler descends into the wrong subtree.

NearestChild under the contract checker: True <- exactness is not what this checks


In [17]:
# The other cases it catches.
class ShapeBug(Verifier):
    name = "shape-bug"
    def verify(self, request):
        return VerifyResult(request.target_mean[:2], accepted=False)


class NaNBug(Verifier):
    name = "nan-bug"
    def verify(self, request):
        return VerifyResult(request.target_mean * np.nan, accepted=False)


class IndexBug(Verifier):
    name = "index-bug"
    def verify(self, request):
        return VerifyResult(request.child(0), accepted=True, child_index=99)


for rule in (ShapeBug, NaNBug, IndexBug):
    try:
        CheckedVerifier(rule())(request)
    except Exception as exc:
        print(f"{rule.name:>10}: {type(exc).__name__} -> {str(exc).splitlines()[0]}")

 shape-bug: ValueError -> shape-bug returned state of shape (2,), expected (6,)
   nan-bug: FloatingPointError -> nan-bug returned a non-finite state
 index-bug: IndexError -> index-bug accepted child_index=99 with K=3


## 7. Declaring the topology a rule supports

A single-proposal coupling sets `max_children = 1`. `check_topology` then refuses a branching
tree **at construction time**, rather than silently ignoring siblings at every node. The sampler
calls it once, so misconfiguration fails at build time.

In [18]:
class SingleProposalRule(Verifier):
    name = "single-proposal"
    max_children = 1

    def verify(self, request):
        return ResampleVerifier().verify(request)


rule = SingleProposalRule()
print("supports K=1 / K=3:", rule.supports(1), rule.supports(3))

rule.check_topology(DraftTree.chain(4))          # fine: K = 1
try:
    rule.check_topology(DraftTree.uniform(branching=3, lookahead=2))
except ValueError as exc:
    print("ValueError ->", exc)

supports K=1 / K=3: True False
ValueError -> SingleProposalRule supports at most K=1 proposals per node, but the draft tree has K=3. Use DraftTree.chain(L) for single-proposal rules.


## 8. The registry

Rules can be addressed by name, which is what a config file needs.

In [19]:
@register_verifier("resample-copy")
class MyRegisteredRule(ResampleVerifier):
    pass


print("available:", available_verifiers())
print("created:  ", type(create_verifier("resample-copy")).__name__, "with name", create_verifier("resample-copy").name)

available: ('d-grs', 'resample', 'resample-copy', 'rmc')
created:   MyRegisteredRule with name resample-copy


In [20]:
rmc, dgrs = create_verifier("rmc"), create_verifier("d-grs")


def node(seed, num_children):
    """A fresh node with its own generator, so each line below is reproducible alone."""
    g = np.random.default_rng(seed)
    return VerifyRequest(
        step=7, proposal_mean=mu_p, target_mean=mu_q, sigma=sigma,
        children=mu_p + sigma * g.standard_normal((num_children, dim)),
        parent_state=np.zeros(dim), rng=g,
    )


def show(rule, seed, num_children):
    out = rule.verify(node(seed, num_children))
    print(f"  seed {seed:>2}: accepted={out.accepted!s:<5} child_index={out.child_index!s:<4} "
          f"examined={out.proposals_examined}")


print("rmc  (K = 1: accept the child, or reflect it)")
for seed in (0, 5):
    show(rmc, seed, 1)

print("\nd-grs (K = 3: sweep in drafting order, stop at the first acceptance,")
print("       and fall back to the residual if none of the three is taken)")
for seed in (0, 16, 26, 5):
    show(dgrs, seed, 3)

print("\nacceptance rises with K -- this is what a tree buys (Theorem 2, eqs. 14-15):")
for num_children in (1, 2, 4, 8):
    r = check_exactness(dgrs, delta=1.0, num_children=num_children, seed=1, num_samples=20000)
    tail = "   <- collapses to eq. (16), RMC's ceiling" if num_children == 1 else ""
    print(f"  d-grs K={num_children}: {r.acceptance_rate:.4f}{tail}")
rmc_rate = check_exactness(rmc, delta=1.0, num_children=1, seed=1, num_samples=20000)
print(f"  rmc   K=1: {rmc_rate.acceptance_rate:.4f}"
      f"   [2 * Phi_bar(delta/2) = {2.0 * standard_normal_sf(0.5):.4f}]")

try:
    rmc.check_topology(DraftTree.uniform(branching=3, lookahead=2))
except ValueError as exc:
    print("\nrmc   on a branching tree -> ValueError:", str(exc).splitlines()[0])
dgrs.check_topology(DraftTree.uniform(branching=3, lookahead=2))
print("d-grs on the same tree    -> fine, max_children is None")

rmc  (K = 1: accept the child, or reflect it)
  seed  0: accepted=True  child_index=0    examined=1
  seed  5: accepted=False child_index=None examined=1

d-grs (K = 3: sweep in drafting order, stop at the first acceptance,
       and fall back to the residual if none of the three is taken)
  seed  0: accepted=True  child_index=0    examined=1
  seed 16: accepted=True  child_index=1    examined=2
  seed 26: accepted=True  child_index=2    examined=3
  seed  5: accepted=False child_index=None examined=4

acceptance rises with K -- this is what a tree buys (Theorem 2, eqs. 14-15):


  d-grs K=1: 0.6106   <- collapses to eq. (16), RMC's ceiling


  d-grs K=2: 0.7092


  d-grs K=4: 0.7940


  d-grs K=8: 0.8639


  rmc   K=1: 0.6228   [2 * Phi_bar(delta/2) = 0.6171]

rmc   on a branching tree -> ValueError: ReflectionMaximalCoupling supports at most K=1 proposals per node, but the draft tree has K=3. Use DraftTree.chain(L) for single-proposal rules.
d-grs on the same tree    -> fine, max_children is None


## 9. Batching comes for free

Every rule gets `verify_batch`, whose default implementation splits the batch and calls your
`verify` row by row — so a rule written against the scalar contract runs under
`BatchedSpeculativeSampler` unchanged. `request.row(j)` does the translation, including turning
the batch-wide `info["nodes"]` back into this row's `info["node"]`.

Override `verify_batch` only when the per-node work is worth vectorising, and keep it
**row-independent**: row `j` may depend only on `request.row(j)`. See the batched tutorial.

In [21]:
from specdiff import BatchedVerifyRequest

batched = BatchedVerifyRequest(
    steps=(3, 9),                                    # rows sit at *different* steps
    indices_in_batch=(0, 4),                         # ... and belong to different images
    proposal_mean=np.stack([mu_p, mu_p]),
    target_mean=np.stack([mu_q, mu_q + 0.2]),
    sigmas=(0.5, 0.31),                              # so sigma is per row, not scalar
    children=np.stack([children, children + 0.1]),   # (batch, K, *state_shape)
    parent_state=np.zeros((2, dim)),
    rng=rng,
    info={"level": 1, "nodes": (0, 2)},
)

row = batched.row(1)
print("batch_size / K:", batched.batch_size, batched.num_children)
print("row(1): step", row.step, " image", row.index_in_batch,
      " sigma", row.sigma, " info", dict(row.info))

out = DeltaProbe().verify_batch(batched)             # the default row-wise loop
print("states:", out.states.shape, " accepted:", out.accepted, " child_index:", out.child_index)

batch_size / K: 2 3
row(1): step 9  image 4  sigma 0.31  info {'level': 1, 'node': 2}
states: (2, 6)  accepted: (False, False)  child_index: (None, None)


## 10. The paper's two rules

`specdiff/verifiers/rmc.py` and `specdiff/verifiers/dgrs.py` hold Algorithms 1 and 2. Together
they are the argument for the whole template: neither touches the sampler, neither imports a
framework, and neither needs a `verify_batch` override to run under batching. They differ in
exactly the two things the paper varies — the draft topology and the verification rule.

**Algorithm 1 — RMC, `K = 1`.** Project the single child to `s_hat`; accept with probability
`1 ^ phi(s_hat - delta) / phi(s_hat)`; on rejection reflect about the crossing point of the two
densities, `s = delta - s_hat`, carrying the child's `z_perp` through untouched. The accepted
branch contributes `min(p, q)` and the reflected branch `(q - p)_+`, which sum to `q` — that is
the exactness argument in one line. Acceptance is `2 * Phi_bar(delta / 2)`, eq. (16), which is
the overlap `1 - TV(P, Q)`: the most *any* single-proposal coupling can achieve.

**Algorithm 2 — D-GRS, any `K`.** Sweep the children **in drafting order** — the sequence
coupling, not the list coupling. Maintain the level `lambda_k` and the residual mass `G_{k+1}`,
accepting child `k` with probability `1 ^ (rho(s_k) - lambda_{k-1})_+ / G_k` where
`rho(s) = phi(s - delta) / phi(s)`. On rejection the level rises to
`lambda_k = lambda_{k-1} + G_k`, inducing the super-level set `H_k = {s : rho(s) >= lambda_k}`
and leaving `G_{k+1} = Q(H_k) - lambda_k P(H_k)`. After `K` rejections it samples the normalised
residual, eq. (13), which is what restores exactness (Theorem 1). Acceptance is `1 - G_{K+1}`
(Theorem 2) and rises with `K`, as the cell above measures.

Because `H_k` is a half-line in the projected coordinate, both masses are `Phi_bar` evaluations
(Appendix B.2) — and eq. (13)'s positive part is a half-line too, so its CDF is a difference of
`Phi_bar`s and inverting it is a bisection. There is no quadrature anywhere in either rule, and
neither ever forms a Gaussian density: `phi` cancels out of RMC's ratio entirely, which is why
`specdiff.ops` ships `Phi` and `Phi_bar` and no PDF.

Four things in those bodies worth stealing for a rule of your own:

* **Project every child before the sweep.** D-GRS's residual branch needs `z_perp` of the
  *first* child, not the last one examined, so the projections cannot be consumed as you go.
* **Accept on `u < beta`, not `u <= beta`.** `ops.uniform` returns `[0, 1)`, so `u` can be
  exactly `0` and `<=` would accept even at `beta = 0`. On the torch backend `torch.rand`
  is float32, so that misfires with probability `2^-24` (~`6e-8`), not `2^-53`.
* **In log space use `math.log1p(-u)`, not `math.log(u)`** — same reason, from the other end:
  `math.log(0)` raises. `1 - u` is uniform too, and `log1p` is defined on precisely the range
  `uniform()` guarantees.
* **Clamp `G_k` at zero.** It is a difference of two survival functions and can go a few ulps
  negative once the remaining mass runs out.

Three traps, the first two specific to a sequence coupling — at `K = 1` there is no order to get
wrong and only one residual to carry, so RMC gets both for free. Keep the children **in order**;
carry through the right orthogonal residual (`Z_perp,k` on acceptance of child `k`, but
`Z_perp,1` on the residual branch); and check `frame.degenerate` before computing any `tau`.

`tests/test_rmc.py` and `tests/test_dgrs.py` are the shape your own tests should take. Note that
exactness alone is a weak check — `ResampleVerifier` passes every KS test and accepts nothing —
so it is the acceptance-probability assertion that pins a rule to the algorithm it claims to be.

The ceiling a single-proposal coupling works towards, from `delta` alone:

In [22]:
print(f"{'delta':>7}{'acceptance alpha':>20}{'chain ceiling':>16}")
for delta in (0.0, 0.1, 0.25, 0.5, 1.0, 2.0):
    alpha = 2.0 * standard_normal_sf(delta / 2.0)      # eq. (16)
    ceiling = "inf" if alpha >= 1.0 else f"{1.0 / (1.0 - alpha):.2f}x"
    print(f"{delta:>7.2f}{alpha:>20.3f}{ceiling:>16}")
print()
print("(delta = 0 is the perfect proposal: every level accepts, so a round is bounded only by L.)")

  delta    acceptance alpha   chain ceiling
   0.00               1.000             inf
   0.10               0.960          25.08x
   0.25               0.901          10.05x
   0.50               0.803           5.07x
   1.00               0.617           2.61x
   2.00               0.317           1.46x

(delta = 0 is the perfect proposal: every level accepts, so a round is bounded only by L.)


The ceiling is `1 / (1 - alpha)`: a round accepts a `Geom(alpha)` prefix of mean
`alpha / (1 - alpha)` and then commits one more state on the rejection (Appendix D.1). The
sampler tutorial measures the real thing against it.

### Checklist for a new rule

- [ ] `verify` returns an exact draw from `N(target_mean, sigma^2 I)`
- [ ] accepted ⟹ the returned state *is* `children[child_index]`
- [ ] children examined in the given order, if sequence-coupled
- [ ] `frame.degenerate` checked before any division by `delta`
- [ ] all randomness drawn from `request.rng`
- [ ] `reset()` clears per-run state
- [ ] `max_children` set if the rule is single-proposal
- [ ] `check_exactness` passes across a sweep of `delta` and `K`, with a fixed seed
- [ ] a run with `check_contract=True` completes
- [ ] any `verify_batch` override agrees with the row-wise loop

**Next:** [`sampler_tutorial.ipynb`](sampler_tutorial.ipynb) — the sampler that calls all of this.